# **1. Disciplinary-event extraction**

This notebook transforms the card sections of the match reports into individual events. Timed rows, halftime incidents, and post-match records are recognized separately.

The extracted content remains in `raw_motive` for auditing. Text columns used in the analysis are produced only after document artifacts and incidental metadata have been removed.

In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pdfplumber
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INDEX_PATH = ROOT / "data" / "raw" / "match-reports" / "match_reports_index.csv"
RECORDS_PATH = ROOT / "data" / "interim" / "disciplinary_records.csv"
API_CACHE_PATH = ROOT / "data" / "interim" / "game_api_cache"
PARSED_PATH = ROOT / "data" / "interim" / "parsed_card_events.csv"
PARSE_METADATA_PATH = ROOT / "data" / "interim" / "parsed_card_events.json"
OUT = ROOT / "data" / "processed" / "card_events.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

index = pd.read_csv(INDEX_PATH)
api_records = pd.read_csv(RECORDS_PATH)
print(f"{len(index):,} indexed matches | {index.pdf_path.notna().sum():,} local PDFs")


2,660 indexed matches | 2,570 local PDFs


In [2]:
def normalize(value: str) -> str:
    value = "".join(character for character in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(character))
    return re.sub(r"[^a-z0-9]+", " ", value.casefold()).strip()


STOPWORDS_PT = frozenset({
    "a", "ao", "aos", "aquela", "aquelas", "aquele", "aqueles", "aquilo",
    "as", "com", "da", "das", "de", "dela", "delas", "dele", "deles",
    "do", "dos", "e", "ela", "elas", "ele", "eles", "em", "entre",
    "essa", "essas", "esse", "esses", "esta", "estas", "este", "estes",
    "eu", "foi", "foram", "ha", "isso", "isto", "ja", "lhe", "lhes",
    "mas", "me", "mesma", "mesmo", "na", "nas", "no", "nos", "o",
    "os", "ou", "para", "pela", "pelas", "pelo", "pelos", "por",
    "qual", "quais", "que", "quem", "se", "seu", "seus", "sua",
    "suas", "um", "uma", "uns", "umas"
})


ROLE_CODES = {
    "AT": "assistant_coach",
    "TC": "coach",
    "TG": "goalkeeper_coach",
    "PF": "fitness_coach",
    "MD": "doctor",
    "FI": "physiotherapist",
    "MA": "masseur",
}


CLOCK_CARD_LINE = re.compile(r"^(?P<clock>\+?\d{1,3}(?::\d{2})?)\s+(?P<period>1T|2T|INT)\s+(?P<number>\S+)\s+(?P<tail>.+)$", re.IGNORECASE)
POST_MATCH_CARD_LINE = re.compile(r"^-\s*PJ\s+(?P<number>\S+)\s+(?P<tail>.+)$", re.IGNORECASE)
INTERVAL_CARD_LINE = re.compile(r"^-\s*INT\s+(?P<number>\S+)\s+(?P<tail>.+)$", re.IGNORECASE)
SECTION = re.compile(r"^Cartões\s+(Amarelos|Vermelhos)", re.IGNORECASE)
END_SECTION = re.compile(r"^(Ocorrências|Substituições|Gols|Motivo de atraso|Observações|Relação de jogadores)", re.IGNORECASE)
FULL_MOTIVE_MARKER = re.compile(r"(?:(?:Cartão\s+Vermelho\s+Direto|2[ºo°]\s*Cartão\s*Amarelo)\s+)?Motivo\s*:", re.IGNORECASE)
MOTIVE_MARKER = re.compile(r"\bMotivo\s*:", re.IGNORECASE)
DUPLICATE_MOTIVE_PREFIX = re.compile(r"Motivo\s*:.*?\s+-\s+Motivo\s*:", re.IGNORECASE)
FOOTER_START = re.compile(r"\b(?:Confedera\w*|Publica[cç][aã]o\s+da\s+S[uú]mula|Emiss[aã]o\s+desta\s+via)\b", re.IGNORECASE)
PAGE_MARKER = re.compile(r"\b(?:P[aá]gina\s*)?\d+\s*/\s*\d+\b", re.IGNORECASE)
ROLE_LEGEND = re.compile(r"(?<!\w)(?:AT|TC|TG|PF|MD|FI|MA)\s*-\s*(?:Assistente\s+T[eé]cnico|T[eé]cnico|Treinador\s+de\s+Goleiros|Preparador\s+F[ií]sico|M[eé]dico|Fisioterapeuta|Massagista)", re.IGNORECASE)
TABLE_LINE = re.compile(r"^(?:Tempo\s+1T/2T|N[ÃA]O\s+HOUVE|AT\s*-\s*Assistente|TC\s*-\s*T[eé]cnico|TG\s*-\s*Treinador|PF\s*-\s*Preparador)", re.IGNORECASE)
CLOCK_ONLY = re.compile(r"^\+?\d{1,3}(?::\d{2})?$")
PERIOD_ONLY = re.compile(r"^(?:1T|2T|INT)$", re.IGNORECASE)
TEAM_GENERIC_TOKENS = frozenset({"clube", "esporte", "futebol", "regatas", "saf", "fc", "ec", "da", "de", "do", "e"})


def remove_stopwords(value: str) -> str | None:
    if pd.isna(value):
        return None
    tokens = [token for token in normalize(value).split() if token not in STOPWORDS_PT and not token.isdigit()]
    return " ".join(tokens) or None


def extract_pdf_text(path: Path) -> str:
    with pdfplumber.open(path) as pdf:
        return "\n".join((page.extract_text() or "") for page in pdf.pages)


def compact_spaces(value: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value))).strip()


def strip_document_artifacts(value: str | None) -> str | None:
    if value is None or pd.isna(value):
        return None
    value = compact_spaces(value)
    footer = FOOTER_START.search(value)
    if footer:
        value = value[:footer.start()]
    value = PAGE_MARKER.sub(" ", value)
    value = ROLE_LEGEND.sub(" ", value)
    value = re.sub(r"\s+([,.;:!?])", r"\1", value)
    value = compact_spaces(value).strip(" -|")
    return value or None


def team_match_score(reported: str, team: str, state: str | None = None) -> float:
    reported_tokens = [token for token in normalize(reported).split() if token not in TEAM_GENERIC_TOKENS]
    team_tokens = [token for token in normalize(team).split() if token not in TEAM_GENERIC_TOKENS]
    if not reported_tokens or not team_tokens:
        return 0.0
    overlap = len(set(reported_tokens) & set(team_tokens))
    score = overlap / len(set(reported_tokens))
    state = normalize(state or "")
    team_norm = normalize(team)
    if reported_tokens == ["atletico"] and state == "mg":
        score += 1.0 if team_norm == "atletico mineiro" else -1.0
    if reported_tokens == ["atletico"] and state == "go":
        score += 1.0 if team_norm == "atletico" or "goianiense" in team_norm else -1.0
    return score


def split_player_team(tail: str, home: str, away: str):
    tail = re.sub(r"\s+/\s+", "/", tail).strip()
    tail = re.sub(r"\s+(?:Cartão\s+Vermelho\s+Direto|2[ºo°]\s*Cartão\s*Amarelo)\s*$", "", tail, flags=re.IGNORECASE)
    teams = [team for team in (home, away) if isinstance(team, str)]

    explicit = re.match(r"^(?P<player>.+?)\s+-\s+(?P<club>.+?)(?:/(?P<state>[A-Z]{2,3}))?$", tail, re.IGNORECASE)
    if explicit and teams:
        ranked = sorted(
            ((team_match_score(explicit.group("club"), team, explicit.group("state")), team) for team in teams),
            reverse=True,
        )
        if ranked[0][0] > 0 and (len(ranked) == 1 or ranked[0][0] > ranked[1][0]):
            return explicit.group("player").strip(" -"), ranked[0][1]

    for team in teams:
        pattern = re.compile(rf"\s*-?\s*{re.escape(team)}(?:/[A-Z]{{2,3}})?\s*$", re.IGNORECASE)
        match = pattern.search(tail)
        if match:
            return tail[:match.start()].strip(" -"), team

    state_suffix = re.match(r"^(?P<body>.+?)/(?P<state>[A-Z]{2,3})$", tail, re.IGNORECASE)
    if state_suffix and teams:
        body = state_suffix.group("body").strip()
        tokens = list(re.finditer(r"[^\W_]+", body, re.UNICODE))
        corporate = {"s", "a", "f", "saf", "fc", "ec"}
        while tokens and normalize(tokens[-1].group()) in corporate:
            tokens.pop()
        if tokens:
            suffix_token = normalize(tokens[-1].group())
            candidates = []
            for team in teams:
                distinctive = {token for token in normalize(team).split() if token not in TEAM_GENERIC_TOKENS}
                if suffix_token in distinctive:
                    candidates.append((team_match_score(suffix_token, team, state_suffix.group("state")), team))
            candidates.sort(reverse=True)
            if candidates and (len(candidates) == 1 or candidates[0][0] > candidates[1][0]):
                return body[:tokens[-1].start()].strip(" -"), candidates[0][1]

    normalized_tail = normalize(tail)
    ranked = sorted(((team_match_score(normalized_tail, team), team) for team in teams), reverse=True)
    if ranked and ranked[0][0] > 0 and (len(ranked) == 1 or ranked[0][0] > ranked[1][0]):
        team = ranked[0][1]
        anchors = [token for token in normalize(team).split() if token not in TEAM_GENERIC_TOKENS]
        for anchor in sorted(anchors, key=len, reverse=True):
            match = re.search(rf"\s*-?\s*{re.escape(anchor)}(?:\W+(?:s\W*a\W*f|fc|ec))?(?:/[A-Z]{{2,3}})?\s*$", normalize(tail), re.IGNORECASE)
            if match:
                raw_match = re.search(rf"\s*-?\s*{re.escape(anchor)}(?:\W+(?:s\W*a\W*f|fc|ec))?(?:/[A-Z]{{2,3}})?\s*$", tail, re.IGNORECASE)
                if raw_match:
                    return tail[:raw_match.start()].strip(" -"), team
    return tail.strip(" -"), None


def coalesce_card_header_fragments(lines: list[str]) -> list[str]:
    lines = [line for line in lines if line and not PAGE_MARKER.fullmatch(line)]
    output = []
    index = 0
    while index < len(lines):
        if (
            index + 3 < len(lines)
            and CLOCK_ONLY.fullmatch(lines[index])
            and PERIOD_ONLY.fullmatch(lines[index + 1])
            and re.fullmatch(r"\S+", lines[index + 2])
            and not SECTION.match(lines[index + 3])
            and not END_SECTION.match(lines[index + 3])
        ):
            output.append(" ".join(lines[index:index + 4]))
            index += 4
            continue
        output.append(lines[index])
        index += 1
    return output


def parse_motive(raw: str):
    raw_clean = strip_document_artifacts(raw) or ""
    raw_clean = DUPLICATE_MOTIVE_PREFIX.sub("Motivo:", raw_clean)
    is_second_yellow = bool(re.search(r"2[ºo°]\s*Cartão\s*Amarelo", raw_clean, re.IGNORECASE))
    pieces = re.split(r"Motivo\s*:\s*", raw_clean, flags=re.IGNORECASE, maxsplit=1)
    content = pieces[-1].strip()
    if content.startswith("-"):
        official_part, free_text = None, content.lstrip("- ").strip()
    elif " - " in content:
        official_part, free_text = content.split(" - ", 1)
    else:
        official_part, free_text = content, None
    match = re.match(r"^(?P<code>[A-Z]\d+(?:\.\d+)*)\.?\s*(?P<description>.*)$", official_part or "")
    if match:
        code = match.group("code")
        description = match.group("description").strip() or None
    else:
        code = None
        description = (official_part or "").strip(" -") or None
    return code, strip_document_artifacts(description), strip_document_artifacts(free_text), is_second_yellow


def parse_header(line: str, metadata: dict, section: str):
    marker = FULL_MOTIVE_MARKER.search(line)
    header = line[:marker.start()].strip() if marker else line
    inline_motive = line[marker.start():].strip() if marker else None
    match = CLOCK_CARD_LINE.match(header)
    post_match = False
    interval = False
    if not match:
        match = POST_MATCH_CARD_LINE.match(header)
        post_match = match is not None
    if not match:
        match = INTERVAL_CARD_LINE.match(header)
        interval = match is not None
    if not match:
        return None, None
    info = match.groupdict()
    player, team = split_player_team(info["tail"], metadata.get("home_team"), metadata.get("away_team"))
    number = info["number"].upper()
    role = ROLE_CODES.get(number, "player")
    clock = None if post_match or interval else info.get("clock")
    period = "PJ" if post_match else ("INT" if interval else info.get("period").upper())
    minute = np.nan if post_match or interval else int(clock.replace("+", "").split(":")[0])
    event = {
        **metadata,
        "reported_card_type": section,
        "clock": clock,
        "period": period,
        "minute": minute,
        "shirt_number": number if role == "player" else None,
        "participant_role": role,
        "player": player,
        "team": team,
        "raw_card_line": line,
        "parse_status": "post_match" if post_match else ("interval" if interval else "parsed"),
        "motive_parts": [],
    }
    return event, inline_motive


def parse_cards(text: str, metadata: dict):
    lines = coalesce_card_header_fragments([compact_spaces(line) for line in text.splitlines()])
    section = None
    current = None
    rows = []
    issues = []

    def flush():
        nonlocal current
        if current and current.get("motive_parts"):
            raw_motive = compact_spaces(" ".join(current.pop("motive_parts")))
            code, official, reason, second = parse_motive(raw_motive)
            current.update({
                "official_code": code,
                "official_reason": official,
                "reason_raw": reason,
                "raw_motive": raw_motive,
                "disciplinary_outcome": "second_yellow" if second else ("yellow" if current["reported_card_type"] == "yellow" else "direct_red"),
                "source_event_number": len(rows) + 1,
            })
            rows.append(current)
        elif current:
            issues.append({"game_id": metadata["game_id"], "issue": "card_without_motive", "line": current.get("raw_card_line")})
        current = None

    for line in lines:
        if not line:
            continue
        footer = FOOTER_START.search(line)
        if footer:
            prefix = line[:footer.start()].strip()
            if current and prefix and not PAGE_MARKER.fullmatch(prefix):
                current["motive_parts"].append(prefix)
            flush()
            section = None
            continue

        section_match = SECTION.match(line)
        if section_match:
            flush()
            section = "yellow" if section_match.group(1).casefold().startswith("amare") else "red"
            continue
        if section and END_SECTION.match(line):
            flush()
            section = None
            continue
        if not section:
            continue
        if TABLE_LINE.match(line):
            continue
        if PAGE_MARKER.fullmatch(line):
            continue

        line = PAGE_MARKER.sub(" ", line)
        header, inline_motive = parse_header(line, metadata, section)
        if header:
            flush()
            current = header
            if inline_motive:
                current["motive_parts"].append(inline_motive)
            continue

        if current and MOTIVE_MARKER.search(line) and any(MOTIVE_MARKER.search(part) for part in current["motive_parts"]):
            flush()
            current = {
                **metadata,
                "reported_card_type": section,
                "clock": None,
                "period": "UNK",
                "minute": np.nan,
                "shirt_number": None,
                "participant_role": "unknown",
                "player": None,
                "team": None,
                "raw_card_line": None,
                "parse_status": "orphan_motive",
                "motive_parts": [line],
            }
            issues.append({"game_id": metadata["game_id"], "issue": "orphan_motive", "line": line})
            continue
        if current and (MOTIVE_MARKER.search(line) or current["motive_parts"]):
            current["motive_parts"].append(line)

    flush()
    return rows, issues


In [3]:
parser_signature = hashlib.sha256(("rigorous-parser-v3|" + "|".join(index.pdf_path.fillna("").astype(str))).encode()).hexdigest()
metadata_columns = ["game_id", "match_date", "year", "competition", "category", "round", "home_team", "away_team", "referee"]

cached_signature = None
if PARSED_PATH.exists() and PARSE_METADATA_PATH.exists():
    cached_signature = pd.read_json(PARSE_METADATA_PATH, typ="series").get("parser_signature")
if cached_signature == parser_signature:
    cards = pd.read_csv(PARSED_PATH)
    issues = []
    errors = []
else:
    events = []
    issues = []
    errors = []
    for row in tqdm(index.itertuples(index=False), total=len(index), desc="Extracting match reports"):
        if not isinstance(row.pdf_path, str):
            continue
        path = ROOT / row.pdf_path
        if not path.exists():
            continue
        metadata = {column: getattr(row, column) for column in metadata_columns}
        try:
            parsed, parsed_issues = parse_cards(extract_pdf_text(path), metadata)
            events.extend(parsed)
            issues.extend(parsed_issues)
        except Exception as exc:
            errors.append({"game_id": row.game_id, "error": repr(exc)})
    cards = pd.DataFrame(events)
    cards.to_csv(PARSED_PATH, index=False)
    pd.Series({"parser_signature": parser_signature}).to_json(PARSE_METADATA_PATH)
missing_team = cards["team"].isna() & cards["player"].notna()
if missing_team.any():
    repaired = cards.loc[missing_team].apply(
        lambda row: split_player_team(row["player"], row["home_team"], row["away_team"]), axis=1
    )
    cards.loc[missing_team, "player"] = repaired.map(lambda pair: pair[0]).to_numpy()
    cards.loc[missing_team, "team"] = repaired.map(lambda pair: pair[1]).to_numpy()
    cards.to_csv(PARSED_PATH, index=False)
motive_fields = cards.raw_motive.map(parse_motive).apply(pd.Series)
motive_fields.columns = ["official_code", "official_reason", "reason_raw", "is_second_yellow"]
cards[["official_code", "official_reason", "reason_raw"]] = motive_fields[["official_code", "official_reason", "reason_raw"]]
cards["disciplinary_outcome"] = np.where(motive_fields.is_second_yellow, "second_yellow", np.where(cards.reported_card_type.eq("yellow"), "yellow", "direct_red"))
parse_issues = pd.DataFrame(issues)
parse_errors = pd.DataFrame(errors)
print(f"{len(cards):,} extracted events | {cards.parse_status.eq('post_match').sum():,} post-match | {len(issues):,} warnings | {len(errors):,} PDF errors")
display(cards.head(3).T)


Extracting match reports:   0%|          | 0/2660 [00:00<?, ?it/s]

14,146 extracted events | 163 post-match | 0 warnings | 0 PDF errors


,0,1,2
game_id,815072,815072,815072
match_date,09/08/2020,09/08/2020,09/08/2020
year,2020,2020,2020
competition,Campeonato Brasileiro,Campeonato Brasileiro,Campeonato Brasileiro
category,Série A,Série A,Série A
round,1,1,1
home_team,Flamengo,Flamengo,Flamengo
away_team,Atlético Mineiro,Atlético Mineiro,Atlético Mineiro
referee,Raphael Claus,Raphael Claus,Raphael Claus
reported_card_type,yellow,yellow,yellow


## **2. Integration with API records**

Individual events are matched to structured records by name, team, period, minute, and disciplinary outcome. Post-match records remain valid even when they have no timestamp.

In [4]:
def api_type_matches(outcome, api_result):
    value = normalize(api_result).replace(" ", "")
    if outcome == "yellow":
        return "amarelo" in value and "2amarelo" not in value
    return "vermelho" in value


api_by_game = {int(key): frame.reset_index(drop=True) for key, frame in api_records.groupby("game_id")}


def pair_score(event, candidate):
    type_match = api_type_matches(event.disciplinary_outcome, candidate.result)
    name_score = SequenceMatcher(None, normalize(event.player), normalize(candidate.player)).ratio() if pd.notna(event.player) else 0.0
    team_score = SequenceMatcher(None, normalize(event.team), normalize(candidate.team)).ratio() if pd.notna(event.team) else 0.0
    period_score = float(pd.notna(event.period) and pd.notna(candidate.period) and str(event.period)[0] in str(candidate.period))
    minute_score = float(pd.notna(event.minute) and pd.notna(candidate.minute) and abs(float(event.minute) - float(candidate.minute)) <= 1)
    score = 0.70 * name_score + 0.10 * team_score + 0.08 * period_score + 0.07 * minute_score + 0.05 * type_match
    return score if type_match else score - 1.0


def match_game(frame):
    candidates = api_by_game.get(int(frame.game_id.iloc[0]))
    output = pd.DataFrame(index=frame.index, columns=["api_record_id", "player_id", "player_nickname", "api_player", "api_team", "api_period", "api_minute", "api_shirt_number", "api_match_score", "metadata_source"])
    if candidates is None or candidates.empty:
        return output
    scores = np.array([[pair_score(event, candidate) for candidate in candidates.itertuples(index=False)] for event in frame.itertuples(index=False)])
    row_indices, candidate_indices = linear_sum_assignment(scores, maximize=True)
    complete_game = len(frame) == len(candidates)
    for row_position, candidate_position in zip(row_indices, candidate_indices):
        event = frame.iloc[row_position]
        candidate = candidates.iloc[candidate_position]
        score = float(scores[row_position, candidate_position])
        type_match = api_type_matches(event.disciplinary_outcome, candidate.result)
        recovered = event.parse_status == "orphan_motive" and complete_game
        if not recovered and (score < 0.62 or not type_match):
            output.loc[event.name, "api_match_score"] = score
            continue
        source = "api_recovered" if recovered and type_match else "api_recovered_type_conflict" if recovered else "pdf_matched"
        output.loc[event.name] = [candidate.record_id, candidate.player_id, candidate.player_nickname, candidate.player, candidate.team, candidate.period, candidate.minute, candidate.shirt_number, score, source]
    return output


cards = cards.reset_index(drop=True)
matched = pd.concat([match_game(frame) for _, frame in cards.groupby("game_id", sort=False)]).sort_index()
cards = pd.concat([cards, matched], axis=1)
cards["player_reported"] = cards["player"]
cards["team"] = cards.pop("api_team").fillna(cards["team"])
cards["player"] = cards.pop("api_player").fillna(cards["player"])
recovered = cards.metadata_source.fillna("").str.startswith("api_recovered")
cards.loc[recovered, "period"] = cards.loc[recovered, "api_period"].map({"TN1": "1T", "AC1": "1T", "TN2": "2T", "AC2": "2T", "INT": "INT", "PJ": "PJ"}).fillna(cards.loc[recovered, "period"])
cards.loc[recovered, "minute"] = pd.to_numeric(cards.loc[recovered, "api_minute"], errors="coerce").astype(float).to_numpy()
cards.loc[recovered, "shirt_number"] = cards.loc[recovered, "api_shirt_number"].astype("string").to_numpy()
cards.loc[recovered, "parse_status"] = "recovered_header"
cards["metadata_source"] = cards["metadata_source"].fillna("pdf_unmatched")
print(f"{cards.player_id.notna().sum():,} matched events | {recovered.sum():,} metadata records recovered from the API")


14,131 matched events | 0 metadata records recovered from the API


## **3. Rigorous cleaning of card justifications**

Footers, page markers, structural markers, roles, known names, teams, referees, dates, times, and numbers are removed from the textual representation. The cleaned text preserves the disciplinary action; the raw material remains available for auditing.

In [5]:
DATE_PATTERN = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
CLOCK_PATTERN = re.compile(r"(?<!\w)\+?\d{1,3}:\d{2}(?!\w)")
MINUTE_PATTERN = re.compile(r"\b(?:aos?|com)?\s*\d+\s*(?:min(?:uto)?s?)(?:\s+do\s+(?:primeiro|segundo|\d+[º°o]?)\s+tempo)?\b|\bdo\s+\d+[º°o]?\s+tempo\b", re.IGNORECASE)
NUMBER_REFERENCE = re.compile(r"\b(?:camisa\s+(?:de\s+)?n[uú]mero|n[uú]mero|n(?:[º°o]|r\.?)?)\s*[-.:]?\s*\d+\b", re.IGNORECASE)
STRUCTURAL_MARKER = re.compile(r"(?:-\s*PJ\b|\bMotivo\s*:|\bCart[aã]o\s+Vermelho\s+Direto\b|\b2[ºo°]\s*Cart[aã]o\s+Amarelo\b)", re.IGNORECASE)
HONORIFIC = re.compile(r"\b(?:sr|sra|senhor|senhora)\.?\b", re.IGNORECASE)
ORDINAL_NUMBER = re.compile(r"(?<!\w)\d+\s*[º°o](?!\w)", re.IGNORECASE)
STANDALONE_NUMBER = re.compile(r"(?<!\w)[+-]?\d+(?!\w)")
ACCENT_CLASSES = {"a": "[aáàâãä]", "e": "[eéèêë]", "i": "[iíìîï]", "o": "[oóòôõö]", "u": "[uúùûü]", "c": "[cç]", "n": "[nñ]"}
NAME_TOKEN = r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'.-]*"
# The {1,7}?/{1,6}? quantifiers below must be lazy: without `?`, the regex tries to match
# as many words as possible before checking the stopping lookahead. Because the lookahead
# accepts any comma or period, it would consume entire phrases after "sr. Fulano"
# (for example, "sr. Igor Gomes por ter calcado de forma temeraria," became only " atleta ").
# With `?`, it stops at the first valid point and preserves the rest of the sentence.
#
# That alone was insufficient: "de" was among the connector words allowed inside a name
# (to support "de Souza") but was absent from the lookahead stopword list, so the regex kept
# consuming "de forma temeraria" as if it were part of the name. _TITLE_STOP now also includes
# "de/dos/das" as stopping terms, and _NAME_CONTINUATION prevents a stopword from being consumed
# through the generic {NAME_TOKEN} fallback. This may truncate real names containing "de/dos/das"
# (rare, and only affecting names not already captured by PLAYER_PATTERN/REFEREE_PATTERN, which are
# built from actual rosters)—an acceptable tradeoff to preserve classification keywords such as
# "temeraria" and "braco".
_TITLE_STOP = r"(?:da|do|de|dos|das|por|com|que|ap[oó]s|e)"
_TITLE_PLACEHOLDER = r"(?:atleta|equipe|pessoa)"
_NAME_CONTINUATION = rf"(?!{_TITLE_STOP}\b){NAME_TOKEN}"
TITLED_NAME = re.compile(rf"\b(?:sr|sra|senhor|senhora)\.?\s+(?!{_TITLE_PLACEHOLDER}\b){NAME_TOKEN}(?:\s+(?:de|da|do|dos|das|e|{_NAME_CONTINUATION})){{1,7}}?(?=\s*(?:[,;.]|\b{_TITLE_STOP}\b|$))", re.IGNORECASE)
FUNCTION_NAME = re.compile(rf"(\b(?:delegado\s+da\s+partida|quarto\s+[aá]rbitro|assistente\s+de\s+arbitragem)\b)[\s,;:]+(?:{NAME_TOKEN})(?:\s+(?:de|da|do|dos|das|e|{NAME_TOKEN})){{1,6}}?(?=\s*(?:[,;.]|\b(?:que|por|ap[oó]s)\b|$))", re.IGNORECASE)
FUNCTION_TERM = re.compile(r"\b(?:aa\s*[12]|[aá]rbitro\s+assistente(?:\s*[12])?|assistente\s+de\s+arbitragem|quarto\s+[aá]rbitro|delegado\s+da\s+partida|observador\s+de\s+var|auxiliar\s+t[eé]cnico|preparador\s+f[ií]sico|fisioterapeuta|massagista|m[eé]dico|treinador|t[eé]cnico|avar\s*[12]?|var)\b", re.IGNORECASE)
FUNCTION_CODE = re.compile(r"\b(?:AA[12]?|AR[12]?|AVAR[12]?|VAR|QA|DEL|INSP|AT|TC|TG|PF|MD|FI|MA)\b", re.IGNORECASE)
NUMBERED_NAME = re.compile(rf"\b(?:(?:atleta|jogador|advers[aá]rio)\s+(?:de\s+)?)?(?:camisa\s+(?:de\s+)?n[uú]mero|n[uú]mero|n(?:[º°o]|r\.?)?)\s*[-.:]?\s*\d+\s+(?:o\s+)?(?:(?:sr|sra|senhor|senhora)\.?\s+)?{NAME_TOKEN}(?:\s+(?:de|da|do|dos|das|e|{NAME_TOKEN})){{1,7}}?(?=\s*(?:[,;.]|\b(?:da|do|por|com|que|ap[oó]s|informo|e)\b|$))", re.IGNORECASE)


def flexible_name_pattern(value):
    tokens = []
    for token in normalize(value).split():
        tokens.append("".join(ACCENT_CLASSES.get(character, re.escape(character)) for character in token))
    return r"[\s.\-']+".join(tokens)


def entity_pattern(values):
    cleaned = {normalize(value) for value in values if isinstance(value, str) and len(normalize(value)) >= 5 and len(normalize(value).split()) >= 2}
    patterns = [flexible_name_pattern(value) for value in sorted(cleaned, key=len, reverse=True)]
    alternatives = "|".join(patterns)
    return re.compile(rf"(?<!\w)(?:{alternatives})(?!\w)", re.IGNORECASE) if alternatives else re.compile(r"(?!x)x")


roster_values = []
official_values = []
for cache_path in API_CACHE_PATH.glob("*.json.gz"):
    try:
        with gzip.open(cache_path, "rt", encoding="utf-8") as file:
            game = json.load(file).get("jogo", {})
        for side in ("mandante", "visitante"):
            for athlete in game.get(side, {}).get("atletas", []):
                for field in ("nome", "apelido"):
                    value = re.sub(r"^\s*\d+\s*-\s*", "", str(athlete.get(field, ""))).strip()
                    if value:
                        roster_values.append(value)
        for official in game.get("arbitros", []):
            value = str(official.get("nome", "")).strip()
            if value:
                official_values.append(value)
    except (OSError, json.JSONDecodeError, AttributeError):
        continue

player_values = pd.concat([
    api_records.get("player", pd.Series(dtype=object)),
    api_records.get("player_nickname", pd.Series(dtype=object)),
    pd.Series(roster_values, dtype=object),
    cards["player_reported"],
    cards["player"],
    cards["player_nickname"],
], ignore_index=True)
team_values = pd.concat([cards["home_team"], cards["away_team"], cards["team"]], ignore_index=True)
referee_values = pd.concat([cards["referee"], pd.Series(official_values, dtype=object)], ignore_index=True)

PLAYER_PATTERN = entity_pattern(player_values.dropna())
TEAM_PATTERN = entity_pattern(team_values.dropna())
REFEREE_PATTERN = entity_pattern(referee_values.dropna())


def clean_reason_text(value: str | None) -> str | None:
    value = strip_document_artifacts(value)
    if not value:
        return None
    value = NUMBERED_NAME.sub(" atleta ", value)
    value = REFEREE_PATTERN.sub(" pessoa ", value)
    value = PLAYER_PATTERN.sub(" atleta ", value)
    value = TEAM_PATTERN.sub(" equipe ", value)
    value = DATE_PATTERN.sub(" ", value)
    value = CLOCK_PATTERN.sub(" ", value)
    value = MINUTE_PATTERN.sub(" ", value)
    value = NUMBER_REFERENCE.sub(" ", value)
    value = ORDINAL_NUMBER.sub(" ", value)
    value = STRUCTURAL_MARKER.sub(" ", value)
    value = ROLE_LEGEND.sub(" ", value)
    value = TITLED_NAME.sub(" atleta ", value)
    value = FUNCTION_NAME.sub(" ", value)
    value = FUNCTION_TERM.sub(" ", value)
    value = HONORIFIC.sub(" ", value)
    value = STANDALONE_NUMBER.sub(" ", value)
    value = FUNCTION_CODE.sub(" ", value)
    value = re.sub(r"\b(o|a)\s+atleta\s+atleta\b", r"\1 atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:da|do|de)\s+equipe\s+equipe\b", " da equipe", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:clube|club|equipe)(?:\s+(?:do|da))?\s+equipe(?:\s+fc)?\b", "equipe", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:o|a)\s*[.,;:]+\s*atleta\b", "o atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\bdo\s+o\s+atleta\b", "do atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\bda\s+a\s+atleta\b", "da atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"[,.;:]+\s*[,.;:]+", ", " , value)
    value = re.sub(r"\bpel[oa]\s*[,;:.]*\s*pessoa\b", "por uma pessoa", value, flags=re.IGNORECASE)
    value = re.sub(r"([:;,.!?])(?=[\"'“”])", r"\1 " , value)
    value = re.sub(r"\batleta\s*[,;:.]+\s*atleta\b", "atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:f\.?\s*b\.?\s*p\.?\s*a\.?|s\.?\s*a\.?\s*f\.?|fc|ec)\b", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+([,.;:!?])", r"\1", value)
    value = compact_spaces(value).strip(" -|,;:")
    return value or None


def clean_row_entities(row):
    value = row.reason_reported
    if not isinstance(value, str):
        return value
    for name in (row.player_reported, row.player):
        tokens = normalize(name).split() if isinstance(name, str) else []
        if len(tokens) >= 2:
            first = flexible_name_pattern(tokens[0])
            last = flexible_name_pattern(tokens[-1])
            value = re.sub(rf"(?<!\w){first}(?:[\s.\-']+{NAME_TOKEN}){{0,6}}[\s.\-']+{last}(?!\w)", " atleta ", value, flags=re.IGNORECASE)
            value = re.sub(rf"(?<!\w){flexible_name_pattern(' '.join(tokens[:min(3, len(tokens))]))}(?!\w)", " atleta ", value, flags=re.IGNORECASE)
    for name in (row.player_nickname, row.referee):
        tokens = normalize(name).split() if isinstance(name, str) else []
        if tokens and len(tokens[0]) >= 5:
            value = re.sub(rf"(?<!\w){flexible_name_pattern(tokens[0])}(?!\w)", " atleta " if name == row.player_nickname else " pessoa ", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(o|a)\s+atleta\s+atleta\b", r"\1 atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\bequipe\s+(?:do|da|de)\s+equipe\b", "equipe", value, flags=re.IGNORECASE)
    value = re.sub(r"\batleta\s*[,;:.]+\s*atleta\b", "atleta", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+([,.;:!?])", r"\1", value)
    return compact_spaces(value).strip(" -|,;:") or None


cards["official_reason"] = cards["official_reason"].map(strip_document_artifacts)
cards["reason_reported"] = cards["reason_raw"].map(clean_reason_text)
cards["reason_reported"] = cards.apply(clean_row_entities, axis=1)
cards["reason_reported"] = cards["reason_reported"].fillna(cards["official_reason"])
cards["reason"] = cards["reason_reported"].map(remove_stopwords)
cards["reason_key"] = cards["reason"].fillna(cards["official_reason"].map(remove_stopwords))
cards["event_id"] = [
    hashlib.sha1(f"{row.game_id}|{row.source_event_number}|{row.player_reported}|{row.period}|{row.clock}|{row.reported_card_type}".encode()).hexdigest()[:16]
    for row in cards.itertuples(index=False)
]


## **4. Quality and traceability**

Final checks measure coverage, segmentation, and residual artifacts. The file is saved only when every event has a unique identifier and no document markers remain in the justifications used by the analysis.

In [6]:
residual_patterns = {
    "document_footer": re.compile(r"confedera|publica[cç][aã]o\s+da\s+s[uú]mula|emiss[aã]o\s+desta\s+via", re.IGNORECASE),
    "page_marker": PAGE_MARKER,
    "embedded_motive": MOTIVE_MARKER,
    "role_legend": ROLE_LEGEND,
    "date": DATE_PATTERN,
    "clock": CLOCK_PATTERN,
    "number_reference": NUMBER_REFERENCE,
    "honorific": HONORIFIC,
    "function_reference": FUNCTION_TERM,
    "function_code": FUNCTION_CODE,
    "known_player_name": PLAYER_PATTERN,
    "known_official_name": REFEREE_PATTERN,
}
residuals = pd.Series({
    name: int(cards.reason_reported.fillna("").map(lambda value: bool(pattern.search(value))).sum())
    for name, pattern in residual_patterns.items()
}, name="remaining")
quality = pd.Series({
    "events": len(cards),
    "matches": cards.game_id.nunique(),
    "post_match_events": cards.parse_status.eq("post_match").sum(),
    "orphan_events": cards.parse_status.eq("orphan_motive").sum(),
    "recovered_header_events": cards.parse_status.eq("recovered_header").sum(),
    "api_type_conflicts": cards.metadata_source.eq("api_recovered_type_conflict").sum(),
    "unique_free_texts": cards.reason_key.nunique(),
    "free_text_present_pct": 100 * cards.reason.notna().mean(),
    "official_code_present_pct": 100 * cards.official_code.notna().mean(),
    "team_present_pct": 100 * cards.team.notna().mean(),
    "player_present_pct": 100 * cards.player.notna().mean(),
    "player_id_matched_pct": 100 * cards.player_id.notna().mean(),
}, name="value")

display(quality.to_frame().round(2))
display(residuals.to_frame())
display(cards.groupby(["year", "competition"], dropna=False).size().rename("events").to_frame())

api_red_by_game = api_records.loc[
    api_records["result"].map(lambda value: "vermelho" in normalize(value)), "game_id"
].value_counts().sort_index()
parsed_red_by_game = cards.loc[
    cards["disciplinary_outcome"].isin(["second_yellow", "direct_red"]), "game_id"
].value_counts().sort_index()
sport_red_by_year = cards.loc[
    cards["team"].fillna("").str.startswith("Sport Recife")
    & cards["disciplinary_outcome"].isin(["second_yellow", "direct_red"])
].groupby("year").size().to_dict()

assert cards.event_id.is_unique
assert cards.loc[cards.api_record_id.notna(), "api_record_id"].is_unique
assert cards.team.notna().all()
assert api_red_by_game.equals(parsed_red_by_game)
assert {year: sport_red_by_year.get(year, 0) for year in (2020, 2021, 2025)} == {2020: 7, 2021: 5, 2025: 4}
assert cards.reason_raw.fillna("").str.contains(MOTIVE_MARKER).sum() == 0
assert residuals.eq(0).all()
assert cards.reason_key.notna().all()
assert cards.player.notna().mean() > 0.90

ordered = [
    "event_id", "game_id", "match_date", "year", "competition", "category", "round",
    "home_team", "away_team", "team", "referee", "api_record_id", "player_id", "player", "player_reported", "player_nickname",
    "shirt_number", "participant_role", "period", "minute", "clock", "reported_card_type", "disciplinary_outcome",
    "official_code", "official_reason", "reason_reported", "reason", "reason_key", "parse_status", "metadata_source",
    "source_event_number", "raw_card_line", "raw_motive", "api_match_score",
]
cards[ordered].to_csv(OUT, index=False)
print(f"Saved to {OUT.relative_to(ROOT)}")


,value
events,14146.00
matches,2543.00
post_match_events,163.00
orphan_events,0.00
recovered_header_events,0.00
api_type_conflicts,0.00
unique_free_texts,4225.00
free_text_present_pct,100.00
official_code_present_pct,99.79
team_present_pct,100.00


,remaining
document_footer,0
page_marker,0
embedded_motive,0
role_legend,0
date,0
clock,0
number_reference,0
honorific,0
function_reference,0
function_code,0


,,events
year,competition,
2020,Campeonato Brasileiro,1900
2021,Campeonato Brasileiro,1950
2022,Campeonato Brasileiro,2129
2023,Campeonato Brasileiro,2266
2024,Campeonato Brasileiro,2284
2025,Campeonato Brasileiro,2079
2026,Campeonato Brasileiro,1538


Saved to data/processed/card_events.csv
